In [2]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms, datasets
from torchvision.utils import save_image, make_grid


BATCH_SIZE = 64
IMAGE_SIZE = 64
CHANNELS_IMG = 3
NUM_CLASSES = 38  # PlantVillage standard
LATENT_DIM = 100
EMBED_SIZE = 100
FEATURES_G = 64
FEATURES_D = 64
LR = 2e-4
BETA1 = 0.5
EPOCHS = 10
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT_DIR = "models/plantvillage_outputs"
os.makedirs(OUTPUT_DIR, exist_ok= True)


class Generator(nn.Module):
    def __init__(self, num_classes, latent_dim, embed_size, img_channels, features_g):
        super(Generator, self).__init__()
        self.embed = nn.Embedding(num_classes, embed_size)
        self.net = nn.Sequential(
            nn.ConvTranspose2d(latent_dim + embed_size, features_g * 8, 4, 1, 0, bias= False),
            nn.BatchNorm2d(features_g * 8),
            nn.ReLU(True),

            nn.ConvTranspose2d(features_g * 8, features_g * 4, 4, 2, 1, bias= False),
            nn.BatchNorm2d(features_g * 4),
            nn.ReLU(True),
            
            nn.ConvTranspose2d(features_g * 4, features_g * 2, 4, 2, 1, bias= False),
            nn.BatchNorm2d(features_g * 2),
            nn.ReLU(True),
            
            nn.ConvTranspose2d(features_g * 2, features_g, 4, 2, 1, bias= False),
            nn.BatchNorm2d(features_g),
            nn.ReLU(True),
            
            nn.ConvTranspose2d(features_g, img_channels, 4, 2, 1, bias= False),
            nn.Tanh()
        )

    def forward(self, noise, labels):
        embedding = self.embed(labels).unsqueeze(2).unsqueeze(3) # [N, embed_size, 1, 1]
        x = torch.cat([noise, embedding], dim= 1)
        return self.net(x)


class Discriminator(nn.Module):
    def __init__(self, num_classes, img_channels, features_d, img_size=  64):
        super(Discriminator, self).__init__()
        self.img_size = img_size
        self.embed = nn.Embedding(num_classes, img_size * img_size)
        self.net = nn.Sequential(
            nn.Conv2d(img_channels + 1, features_d, 4, 2, 1, bias= False),
            nn.LeakyReLU(0.2, inplace= True),
            nn.Conv2d(features_d, features_d * 2, 4, 2, 1, bias= False),
            nn.BatchNorm2d(features_d * 2),
            nn.LeakyReLU(0.2, inplace= True),
            nn.Conv2d(features_d * 2, features_d * 4, 4, 2, 1, bias= False),
            nn.BatchNorm2d(features_d * 4),
            nn.LeakyReLU(0.2, inplace= True),
            nn.Conv2d(features_d * 4, features_d * 8, 4, 2, 1, bias= False),
            nn.BatchNorm2d(features_d * 8),
            nn.LeakyReLU(0.2, inplace= True),
            nn.Conv2d(features_d * 8, 1, 4, 1, 0, bias= False),
            nn.Sigmoid()
        )

    def forward(self, x, labels):
        embedding = self.embed(labels).view(-1, 1, self.img_size, self.img_size)
        x = torch.cat([x, embedding], dim= 1) 
        return self.net(x)

In [9]:
transform = transforms.Compose([transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), transforms.ToTensor(), transforms.Normalize([0.5]*CHANNELS_IMG, [0.5]*CHANNELS_IMG)])

DATASET_PATH = "data/PlantVillage" 
dataset = datasets.ImageFolder(root= DATASET_PATH, transform= transform)
loader = DataLoader(dataset, batch_size= BATCH_SIZE, shuffle= True, num_workers= 2)

gen = Generator(NUM_CLASSES, LATENT_DIM, EMBED_SIZE, CHANNELS_IMG, FEATURES_G).to(DEVICE)
disc = Discriminator(NUM_CLASSES, CHANNELS_IMG, FEATURES_D, IMAGE_SIZE).to(DEVICE)

opt_gen = optim.Adam(gen.parameters(), lr= LR, betas= (BETA1, 0.999))
opt_disc = optim.Adam(disc.parameters(), lr= LR, betas= (BETA1, 0.999))
criterion = nn.BCELoss()

In [11]:
for epoch in range(EPOCHS):
    for batch_idx, (real, labels) in enumerate(loader):
        real = real.to(DEVICE)
        labels = labels.to(DEVICE)
        batch_size = real.shape[0]

        noise = torch.randn(batch_size, LATENT_DIM, 1, 1).to(DEVICE)
        fake = gen(noise, labels)
        disc_real = disc(real, labels).view(-1)
        loss_D_real = criterion(disc_real, torch.ones_like(disc_real))
        disc_fake = disc(fake.detach(), labels).view(-1)
        loss_D_fake = criterion(disc_fake, torch.zeros_like(disc_fake))
        loss_D = (loss_D_real + loss_D_fake) / 2
        disc.zero_grad()
        loss_D.backward()
        opt_disc.step()

        output = disc(fake, labels).view(-1)
        loss_G = criterion(output, torch.ones_like(output))
        gen.zero_grad()
        loss_G.backward()
        opt_gen.step()

    print(f"Epoch [{epoch+1}/{EPOCHS}] | Loss D: {loss_D:.4f}, Loss G: {loss_G:.4f}")
    
    if epoch % 2 == 0 or epoch == EPOCHS - 1:
        save_image(fake[:16], f"{OUTPUT_DIR}/sample_epoch_{epoch}.png", nrow= 4, normalize= True)

torch.save(gen.state_dict(), "models/pv_generator.pth")
torch.save(disc.state_dict(), "models/pv_discriminator.pth")
print("DONE.")

Epoch [1/10] | Loss D: 0.3845, Loss G: 1.7180
Epoch [2/10] | Loss D: 0.5625, Loss G: 3.9494
Epoch [3/10] | Loss D: 0.0712, Loss G: 4.6444
Epoch [4/10] | Loss D: 0.1109, Loss G: 4.1593
Epoch [5/10] | Loss D: 0.1101, Loss G: 5.6088
Epoch [6/10] | Loss D: 0.2528, Loss G: 9.9107
Epoch [7/10] | Loss D: 0.1299, Loss G: 3.0245
Epoch [8/10] | Loss D: 0.0997, Loss G: 5.2918
Epoch [9/10] | Loss D: 0.2168, Loss G: 2.0308
Epoch [10/10] | Loss D: 0.2943, Loss G: 9.0971
DONE.
